# 02 — Modèle de référence : TF-IDF + régression logistique
**Objectif :** entraîner un premier modèle et mesurer ses performances sur la validation.

**Prérequis :** exécuter le notebook 01. **Sorties :** modèle, métriques et prédictions sauvegardés pour le notebook 03.

TF-IDF représente les messages par leurs mots et groupes de deux mots. La régression logistique prédit ensuite la catégorie. Les réglages de ton code sont conservés.

In [1]:
from pathlib import Path
import json
import pandas as pd

# Ouvrir Jupyter depuis le dossier projet_banking77.
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"
for directory in (DATA_DIR, RESULTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

from time import perf_counter
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

## 1. Recharger le même découpage

In [2]:
required = [DATA_DIR / name for name in ("train.csv", "validation.csv", "categories.json")]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f"Fichier absent : {path}. Exécuter d'abord le notebook 01.")
train_df = pd.read_csv(DATA_DIR / "train.csv", keep_default_na=False)
val_df = pd.read_csv(DATA_DIR / "validation.csv", keep_default_na=False)
categories = pd.Index(json.loads((DATA_DIR / "categories.json").read_text(encoding="utf-8")))
print(f"Train : {len(train_df)} | Validation : {len(val_df)} | Catégories : {len(categories)}")

Train : 7993 | Validation : 1999 | Catégories : 77


## 2. Entraîner le modèle

In [3]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=42,
    )),
])

start = perf_counter()
baseline.fit(train_df["text"], train_df["category"])
training_seconds = perf_counter() - start

print(f"Temps d'entraînement CPU : {training_seconds:.2f} secondes")
print("Taille du vocabulaire :", len(baseline.named_steps["tfidf"].vocabulary_))
print("Itérations :", baseline.named_steps["classifier"].n_iter_)

Temps d'entraînement CPU : 9.65 secondes
Taille du vocabulaire : 20777
Itérations : [40]


## 3. Évaluer sur la validation
L'**accuracy** mesure la proportion de bonnes prédictions. Le **macro-F1** donne le même poids à chaque catégorie. Les données de test ne servent pas à régler le modèle.

In [4]:
baseline_predictions = baseline.predict(val_df["text"])
baseline_metrics = {
    "accuracy": accuracy_score(val_df["category"], baseline_predictions),
    "macro_f1": f1_score(val_df["category"], baseline_predictions, average="macro", zero_division=0),
    "training_seconds": training_seconds,
}
print(pd.Series(baseline_metrics).round(4).to_string())
print(classification_report(val_df["category"], baseline_predictions, zero_division=0))

accuracy            0.8479
macro_f1            0.8408
training_seconds    9.6500
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       1.00      0.94      0.97        32
                                activate_my_card       0.96      0.84      0.90        32
                                       age_limit       1.00      0.95      0.98        22
                         apple_pay_or_google_pay       1.00      0.96      0.98        25
                                     atm_support       1.00      0.65      0.79        17
                                automatic_top_up       0.96      0.92      0.94        25
         balance_not_updated_after_bank_transfer       0.76      0.91      0.83        34
balance_not_updated_after_cheque_or_cash_deposit       0.87      0.94      0.91        36
                         beneficiary_not_allowed       0.88      0.71      0.79        31
                  

## 4. Examiner les erreurs

In [5]:
validation_predictions = val_df[["text", "category"]].copy()
validation_predictions["prediction"] = baseline_predictions
errors = validation_predictions.loc[validation_predictions["category"] != validation_predictions["prediction"]]
print("Nombre d'erreurs :", len(errors))
errors.head(10)

Nombre d'erreurs : 304


,text,category,prediction
4,Why does my identity need to be verified?,why_verify_identity,unable_to_verify_identity
12,Is there a way you can have money from a purch...,request_refund,transfer_into_account
13,What is the process for unlocking the pin?,pin_blocked,get_physical_card
16,How is this transfer supposed to work? It's no...,failed_transfer,card_not_working
17,"Hi, I have been transferring money to my frien...",failed_transfer,transfer_fee_charged
19,Will a cheque suffice?,top_up_by_cash_or_cheque,balance_not_updated_after_cheque_or_cash_deposit
24,i disagree with a charge,card_payment_not_recognised,extra_charge_on_statement
25,Do I have to pick a creditcard brand?,visa_or_mastercard,transaction_charged_twice
26,Why is your exchange rate different on differe...,exchange_rate,card_payment_wrong_exchange_rate
27,I didn't cancel my top-up.,top_up_reverted,top_up_failed


## 5. Sauvegarder le modèle et les résultats

In [6]:
joblib.dump(baseline, MODELS_DIR / "baseline.joblib")
(RESULTS_DIR / "baseline_metrics.json").write_text(
    json.dumps(baseline_metrics, indent=2), encoding="utf-8"
)
validation_predictions.to_csv(RESULTS_DIR / "baseline_validation_predictions.csv", index=False)
print("Modèle et résultats sauvegardés. Passer au notebook 03.")

Modèle et résultats sauvegardés. Passer au notebook 03.
